# Módulo 4 — Agentes com Ferramentas no Setor Elétrico

Neste notebook construímos um agente de IA com **Function Calling** — o modelo escolhe autonomamente quais ferramentas Python chamar para responder uma pergunta.

## O que vamos construir

Um agente que recebe perguntas em linguagem natural sobre qualidade de dados de consumidores e usa ferramentas reais para respondê-las:

1. `validar_documento` — valida CPF ou CNPJ
2. `classificar_consumo` — detecta anomalia por z-score
3. `verificar_consistencia_tarifaria` — regras ANEEL de modalidade × classe
4. `buscar_consumidor` — consulta o DataFrame local
5. `gerar_resumo_qualidade` — relatório estruturado de um registro

> **Pré-requisito:** `GEMINI_API_KEY` configurada no arquivo `.env` na raiz do projeto.

In [ ]:
import os
import re
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))  # raiz do projeto

try:
    from google import genai
    from google.genai import types
    print("google-genai: OK")
except ImportError:
    print("Instale com: uv add google-genai")
    sys.exit(1)

import pandas as pd
import numpy as np

# Carregar a base de dados simulada
possible = [
    "../../modulo2_dados_com_pandas/dados/consumidores_simulado.csv",
    "../modulo2_dados_com_pandas/dados/consumidores_simulado.csv",
]
csv_path = next((p for p in possible if Path(p).exists()), None)
if csv_path:
    df_base = pd.read_csv(csv_path)
    print(f"Base carregada: {len(df_base)} consumidores")
else:
    # fallback: criar amostra mínima
    df_base = pd.DataFrame({
        "cod_consumidor": range(1, 11),
        "nom_consumidor": [f"Consumidor {i}" for i in range(1, 11)],
        "cpf_cnpj": ["12345678901", None, "12345678000195", "00000000000", "98765432100",
                     None, "11122233344", "12345678901", None, "55566677788"],
        "cod_modalidade_tarifaria": ["B1","B1","A4","B2","B1","B3","B1","B2","B1","B1"],
        "cod_classe_consumo": ["RESIDENCIAL","RESIDENCIAL","INDUSTRIAL","COMERCIAL","RESIDENCIAL",
                               "COMERCIAL","RESIDENCIAL","RESIDENCIAL","RURAL","RESIDENCIAL"],
        "vlr_consumo_medio_kwh": [150, 200, 15000, 800, 180, 350, 9999999, 160, 90, 210],
        "flg_ativo": [True]*10,
    })
    print("Base simulada criada (CSV não encontrado)")

client = genai.Client()
print("Cliente Gemini inicializado.")

## 1. Definindo as Ferramentas (Tools)

Cada ferramenta é uma função Python normal. O Gemini lê a assinatura e a docstring para saber quando e como chamá-la.

In [ ]:
# --- Ferramenta 1: Validar CPF ou CNPJ ---

def validar_documento(documento: str) -> dict:
    """
    Valida se um CPF ou CNPJ é válido, verificando formato e dígitos verificadores.

    Args:
        documento: string com o CPF (11 dígitos) ou CNPJ (14 dígitos),
                   com ou sem formatação (pontos, traços, barras).

    Returns:
        dict com campos: tipo ('CPF'|'CNPJ'|'DESCONHECIDO'), valido (bool), motivo (str).
    """
    if not documento:
        return {"tipo": "DESCONHECIDO", "valido": False, "motivo": "Documento ausente"}

    digitos = re.sub(r"\D", "", str(documento))

    if len(digitos) == 11:
        tipo = "CPF"
        # Rejeita sequências repetidas
        if len(set(digitos)) == 1:
            return {"tipo": tipo, "valido": False, "motivo": "Sequência repetida"}
        # Dígito verificador
        def calc_digito(d, pesos):
            s = sum(int(d[i]) * pesos[i] for i in range(len(pesos)))
            r = s % 11
            return 0 if r < 2 else 11 - r
        valido = (
            calc_digito(digitos, range(10, 1, -1)) == int(digitos[9]) and
            calc_digito(digitos, range(11, 1, -1)) == int(digitos[10])
        )
        return {"tipo": tipo, "valido": valido, "motivo": "OK" if valido else "Dígitos verificadores inválidos"}

    elif len(digitos) == 14:
        tipo = "CNPJ"
        if len(set(digitos)) == 1:
            return {"tipo": tipo, "valido": False, "motivo": "Sequência repetida"}
        def calc_cnpj(d, pesos):
            s = sum(int(d[i]) * pesos[i] for i in range(len(pesos)))
            r = s % 11
            return 0 if r < 2 else 11 - r
        p1 = [5,4,3,2,9,8,7,6,5,4,3,2]
        p2 = [6,5,4,3,2,9,8,7,6,5,4,3,2]
        valido = (
            calc_cnpj(digitos, p1) == int(digitos[12]) and
            calc_cnpj(digitos, p2) == int(digitos[13])
        )
        return {"tipo": tipo, "valido": valido, "motivo": "OK" if valido else "Dígitos verificadores inválidos"}

    return {"tipo": "DESCONHECIDO", "valido": False, "motivo": f"Comprimento inesperado: {len(digitos)} dígitos"}


# --- Ferramenta 2: Detectar anomalia de consumo ---

def classificar_consumo(
    cod_consumidor: int,
    vlr_consumo_kwh: float,
) -> dict:
    """
    Classifica o consumo de uma unidade consumidora comparando com a distribuição da base.

    Usa z-score em relação à média e desvio padrão da base carregada.
    |z| <= 2 = NORMAL, 2 < |z| <= 3 = ATENCAO, |z| > 3 = ANOMALIA.

    Args:
        cod_consumidor: código da UC a analisar.
        vlr_consumo_kwh: consumo médio mensal em kWh.

    Returns:
        dict com status ('NORMAL'|'ATENCAO'|'ANOMALIA'), z_score, media_base, desvio_base.
    """
    consumos = df_base["vlr_consumo_medio_kwh"].dropna()
    media = consumos.mean()
    desvio = consumos.std()

    if desvio == 0:
        return {"status": "NORMAL", "z_score": 0.0, "media_base": media, "desvio_base": desvio}

    z = (vlr_consumo_kwh - media) / desvio

    if abs(z) <= 2:
        status = "NORMAL"
    elif abs(z) <= 3:
        status = "ATENCAO"
    else:
        status = "ANOMALIA"

    return {
        "cod_consumidor": cod_consumidor,
        "status": status,
        "z_score": round(float(z), 2),
        "media_base_kwh": round(float(media), 1),
        "desvio_base_kwh": round(float(desvio), 1),
        "vlr_consumo_kwh": vlr_consumo_kwh,
    }


# --- Ferramenta 3: Verificar consistência tarifária ---

def verificar_consistencia_tarifaria(
    cod_modalidade_tarifaria: str,
    cod_classe_consumo: str,
) -> dict:
    """
    Verifica se a modalidade tarifária é compatível com a classe de consumo, conforme PRODIST/ANEEL.

    Grupo B (baixa tensão): B1=Residencial, B2=Rural, B3=Demais, B4=Iluminação Pública.
    Grupo A (alta tensão): A1..AS para Industrial, Comercial, Poder Público, Serviço Público.

    Args:
        cod_modalidade_tarifaria: ex. 'B1', 'B2', 'A4'.
        cod_classe_consumo: ex. 'RESIDENCIAL', 'INDUSTRIAL', 'COMERCIAL'.

    Returns:
        dict com consistente (bool) e motivo (str).
    """
    regras = {
        "B1": ["RESIDENCIAL", "BAIXA RENDA"],
        "B2": ["RURAL", "COOPERATIVA RURAL"],
        "B3": ["COMERCIAL", "INDUSTRIAL", "SERVICO PUBLICO", "PODER PUBLICO", "OUTROS"],
        "B4": ["ILUMINACAO PUBLICA"],
        "A1": ["INDUSTRIAL", "COMERCIAL", "PODER PUBLICO", "SERVICO PUBLICO"],
        "A2": ["INDUSTRIAL", "COMERCIAL", "PODER PUBLICO", "SERVICO PUBLICO"],
        "A3": ["INDUSTRIAL", "COMERCIAL", "PODER PUBLICO", "SERVICO PUBLICO"],
        "A3A": ["INDUSTRIAL", "COMERCIAL", "PODER PUBLICO", "SERVICO PUBLICO"],
        "A4": ["INDUSTRIAL", "COMERCIAL", "PODER PUBLICO", "SERVICO PUBLICO"],
        "AS": ["INDUSTRIAL", "COMERCIAL", "PODER PUBLICO", "SERVICO PUBLICO"],
    }

    modalidade = cod_modalidade_tarifaria.upper().strip()
    classe = cod_classe_consumo.upper().strip()

    if modalidade not in regras:
        return {"consistente": False, "motivo": f"Modalidade '{modalidade}' não reconhecida pelo PRODIST"}

    classes_validas = regras[modalidade]
    consistente = classe in classes_validas

    return {
        "consistente": consistente,
        "modalidade": modalidade,
        "classe": classe,
        "classes_validas_para_modalidade": classes_validas,
        "motivo": "OK" if consistente else f"Classe '{classe}' incompatível com modalidade '{modalidade}' (PRODIST)",
    }


# --- Ferramenta 4: Buscar consumidor na base ---

def buscar_consumidor(cod_consumidor: int) -> dict:
    """
    Retorna todos os dados cadastrais de um consumidor a partir do seu código.

    Args:
        cod_consumidor: código numérico da unidade consumidora.

    Returns:
        dict com os dados do consumidor, ou mensagem de erro se não encontrado.
    """
    registro = df_base[df_base["cod_consumidor"] == cod_consumidor]
    if registro.empty:
        return {"erro": f"Consumidor {cod_consumidor} não encontrado na base."}
    return registro.iloc[0].to_dict()


# --- Ferramenta 5: Gerar resumo de qualidade ---

def gerar_resumo_qualidade(cod_consumidor: int) -> dict:
    """
    Executa todas as verificações de qualidade de um consumidor e retorna um resumo consolidado.

    Verifica: existência na base, validade do documento, consumo, consistência tarifária.
    Calcula um score de qualidade de 0 a 100.

    Args:
        cod_consumidor: código da UC a verificar.

    Returns:
        dict com score, lista de problemas encontrados e detalhes por verificação.
    """
    dados = buscar_consumidor(cod_consumidor)
    if "erro" in dados:
        return dados

    problemas = []
    detalhes = {}

    # Documento
    doc_result = validar_documento(str(dados.get("cpf_cnpj", "") or ""))
    detalhes["documento"] = doc_result
    if not doc_result["valido"]:
        problemas.append(f"Documento inválido: {doc_result['motivo']}")

    # Consumo
    consumo = dados.get("vlr_consumo_medio_kwh")
    if consumo is not None:
        consumo_result = classificar_consumo(cod_consumidor, float(consumo))
        detalhes["consumo"] = consumo_result
        if consumo_result["status"] != "NORMAL":
            problemas.append(f"Consumo com {consumo_result['status']} (z={consumo_result['z_score']})")
    else:
        problemas.append("Consumo médio ausente")
        detalhes["consumo"] = {"status": "SEM_DADO"}

    # Tarifa
    tarifa_result = verificar_consistencia_tarifaria(
        str(dados.get("cod_modalidade_tarifaria", "")),
        str(dados.get("cod_classe_consumo", "")),
    )
    detalhes["tarifa"] = tarifa_result
    if not tarifa_result["consistente"]:
        problemas.append(f"Inconsistência tarifária: {tarifa_result['motivo']}")

    score = max(0, 100 - len(problemas) * 25)

    return {
        "cod_consumidor": cod_consumidor,
        "nom_consumidor": dados.get("nom_consumidor"),
        "score_qualidade": score,
        "status_geral": "OK" if not problemas else "PROBLEMAS_ENCONTRADOS",
        "qtd_problemas": len(problemas),
        "problemas": problemas,
        "detalhes": detalhes,
    }


print("5 ferramentas definidas:")
for fn in [validar_documento, classificar_consumo, verificar_consistencia_tarifaria, buscar_consumidor, gerar_resumo_qualidade]:
    print(f"  • {fn.__name__}")

## 2. Testando as ferramentas diretamente

Antes de conectar ao agente, veja que cada ferramenta funciona de forma isolada.

In [ ]:
# Testar cada ferramenta individualmente
print("=== validar_documento ===")
for doc in ["12345678909", "00000000000", None, "12345678000195"]:
    r = validar_documento(str(doc) if doc else "")
    print(f"  {str(doc):<20} → {r['tipo']} | válido={r['valido']} | {r['motivo']}")

print("\n=== classificar_consumo ===")
for cod, kwh in [(1, 150), (7, 9999999), (3, 15000)]:
    r = classificar_consumo(cod, kwh)
    print(f"  UC {cod} | {kwh:>10.0f} kWh → {r['status']} (z={r['z_score']})")

print("\n=== verificar_consistencia_tarifaria ===")
for mod, classe in [("B1", "RESIDENCIAL"), ("B1", "INDUSTRIAL"), ("A4", "COMERCIAL")]:
    r = verificar_consistencia_tarifaria(mod, classe)
    print(f"  {mod} × {classe:<15} → consistente={r['consistente']} | {r['motivo']}")

print("\n=== gerar_resumo_qualidade ===")
resumo = gerar_resumo_qualidade(7)  # outlier proposital
print(f"  UC 7 | Score={resumo['score_qualidade']} | Problemas={resumo['problemas']}")

## 3. O Agente em ação

Agora passamos as 5 ferramentas para o Gemini. Ele decide autonomamente qual chamar, em qual ordem, quantas vezes — e compõe a resposta final.

In [ ]:
ferramentas = [
    validar_documento,
    classificar_consumo,
    verificar_consistencia_tarifaria,
    buscar_consumidor,
    gerar_resumo_qualidade,
]

perguntas = [
    "O consumidor de código 7 tem algum problema de qualidade? Me dê um diagnóstico completo.",
    "Verifique se o CPF '12345678909' é válido e me explique o resultado.",
    "A modalidade B1 é compatível com um consumidor industrial? Por quê?",
    "Busque os dados do consumidor 3 e me diga se o consumo dele é normal para a base.",
]

for pergunta in perguntas:
    print(f"\n{'─'*65}")
    print(f"PERGUNTA: {pergunta}")
    print(f"{'─'*65}")
    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=pergunta,
            config=types.GenerateContentConfig(
                tools=ferramentas,
                temperature=0.1,
            ),
        )
        print(f"RESPOSTA:\n{response.text}")
    except Exception as e:
        print(f"Erro: {e}")

## 4. Agente analisando a base inteira

Agora deixamos o agente livre para identificar os piores registros da base e gerar um relatório de prioridades.

In [ ]:
# Dar ao agente a lista de UCs e deixá-lo priorizar
lista_ucs = df_base["cod_consumidor"].tolist()

prompt_analise = f"""
Você é um analista de qualidade de dados de uma distribuidora de energia elétrica.

Tenho {len(lista_ucs)} unidades consumidoras na base: {lista_ucs}

Sua tarefa:
1. Use a ferramenta gerar_resumo_qualidade para verificar CADA UC da lista
2. Classifique-as por score (do pior para o melhor)
3. Apresente um relatório final com:
   - Top 3 piores UCs (score mais baixo) com seus problemas
   - Quantas UCs estão OK vs. com problemas
   - Principal tipo de problema encontrado
"""

try:
    print("Agente analisando a base completa...")
    print("(Isso pode levar alguns segundos — o modelo fará múltiplas chamadas de ferramentas)\n")
    
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt_analise,
        config=types.GenerateContentConfig(
            tools=ferramentas,
            temperature=0.1,
        ),
    )
    print(response.text)
except Exception as e:
    print(f"Erro: {e}")

## 5. Inspecionando as chamadas de ferramentas

O Gemini registra quais ferramentas chamou. Veja o processo de raciocínio do agente.

In [ ]:
# Fazer uma pergunta simples e inspecionar as function calls
try:
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents="Faça um resumo de qualidade completo do consumidor 7.",
        config=types.GenerateContentConfig(
            tools=ferramentas,
            temperature=0.1,
        ),
    )

    print("Chamadas de ferramentas realizadas pelo agente:\n")
    for i, candidate in enumerate(response.candidates):
        for part in candidate.content.parts:
            if hasattr(part, "function_call") and part.function_call:
                fc = part.function_call
                print(f"  [{i+1}] {fc.name}(")
                for k, v in fc.args.items():
                    print(f"        {k} = {v!r}")
                print(f"       )")

    print(f"\nResposta final:\n{response.text}")
except Exception as e:
    print(f"Erro: {e}")